# Cyclic peptide binders

Design a macrocyclic peptide against a protein target, for each of the three cyclization
chemistries. The target keeps its experimental coordinates and hotspots; the peptide's
structure and sequence are both generated.

| Chemistry | Closing bond | Ideal |
|---|---|---|
| Head-to-tail | C-terminal `C` to N-terminal `N` (amide) | 1.33 A |
| Disulfide | `CYS.SG` to `CYS.SG` | 2.05 A |
| Isopeptide | `LYS.NZ` to `ASP.CG` (amide) | 1.33 A |

**The regime decides the outcome, not the chemistry.** With input coordinates on the ring,
all three close reliably once the crosslinked residues are atomized — see the
[cyclic peptide tutorial](cyclic_peptide_tutorial.ipynb). A *designed* ring has no input
coordinates, and atomized tokens require finite ones at inference, so that fix is
unavailable here. Section 5 shows what this costs.

Target: PD-1 (`01_PD1` from the PPI benchmark), hotspots VAL33, ILE95, LEU97, GLN102.

In [ ]:
from pathlib import Path
import os, subprocess, glob

import numpy as np

REPO_ROOT = Path(os.environ.get("RFD4_REPO", "")).resolve() if os.environ.get("RFD4_REPO") else None
if REPO_ROOT is None:
    here = Path.cwd()
    for cand in (here, *here.parents):
        if (cand / ".project-root").exists():
            REPO_ROOT = cand
            break
assert REPO_ROOT is not None, "Run inside the repository, or set RFD4_REPO."

WORK = Path(os.environ.get("CYCLIC_BINDER_WORK", REPO_ROOT / "scratch" / "cyclic_binder_tutorial"))
WORK.mkdir(parents=True, exist_ok=True)

INPUTS = REPO_ROOT / "docs" / "examples" / "inputs" / "cyclic_binders"
CHEMISTRIES = ["head_tail", "disulfide", "isopeptide"]
HOTSPOTS = [33, 95, 97, 102]          # PD-1 target residues, as annotated in the benchmark
LENGTH = 12                            # ring size; the disulfide's useful band is 10-16

print("repo:", REPO_ROOT)
print("inputs:", [p.name for p in sorted(INPUTS.glob("*.cif.zst"))])

## 1. The input

Two chains: the **target** `B`, with coordinates, full sequence condition and hotspots from
the benchmark, and the **peptide** `P`, with no coordinate conditioning at all. Only the two
residues the chemistry depends on have their identity fixed.

In [ ]:
import rfproteina.conditions  # noqa: F401 -- registers condition classes; see the traps below
from atomworks.io.parser import parse
from atomworks.io.utils.atom_array_plus import as_atom_array_plus
from rfproteina.utils.parse_presets import ANNOTATED_CIF


def load(chemistry):
    path = INPUTS / f"01_PD1_{chemistry}_{LENGTH:02d}.cif.zst"
    return as_atom_array_plus(parse(str(path), config=ANNOTATED_CIF)["assemblies"]["1"][0])


aa = load("head_tail")
pep, tgt = aa.chain_id == "P", aa.chain_id == "B"
seq = np.asarray(aa.get_annotation("condition_sequence_1_residue"))
hot = np.asarray(aa.get_annotation("condition_hotspot_1_atom")).astype(bool)

print(f"peptide chain P : {int(pep.sum()):4d} atoms, {len(set(aa.res_id[pep].tolist()))} residues")
print(f"target  chain B : {int(tgt.sum()):4d} atoms, {len(set(aa.res_id[tgt].tolist()))} residues")
print(f"peptide sequence condition : {sorted(set(seq[pep].tolist()))}   ('<M>' = designed)")
print(f"hotspot atoms on target    : {int(hot.sum())}")
print(f"bonds (incl. the closure)  : {len(aa.bonds.as_array())}")

Three ways to author this wrongly. Each produces **no error** — the run exits
`0` and writes a plausible file.

- **`expseg` bounds default to `-1`.** `-1` means "not an expandable segment"; `0` means
  "expand to zero residues", and `RealizeExpandableSegments` deletes the chain. Generation
  still succeeds, with the peptide absent from the output.
- **`import rfproteina.conditions` is load-bearing.** Without it the condition classes are
  unregistered and the hotspots are dropped on write — the same silent drop as an RCSB
  `disulf` record, which the parse preset also ignores. Declare closures as `covale`.
- **A length-range binder cannot carry a closure bond.** The PPI benchmark specifies binders
  as a placeholder row (`80-150`, one `UNK` atom); `_struct_conn` names two atoms of two
  numbered residues, so the peptide chain has to be enumerated explicitly, as these inputs
  are.

Validate before spending GPU time — look for `struct_conn_ok ... all covale linkages
verified in the bond graph`, and for `struct_conn_bond_missing`:

```bash
bash skills/rfd4-proteina-annotated-cif/scripts/check_annotated_cif.sh <input.cif>
```

## 2. The target

The peptide has no coordinates yet, so there is nothing of it to draw. Hotspots in orange.

In [ ]:
import io
import warnings

import py3Dmol
import biotite.structure.io.pdb as biopdb


def to_pdb(atom_array):
    """Render an AtomArray as a PDB string, which is what the 3Dmol viewers take."""
    f = biopdb.PDBFile()
    with warnings.catch_warnings():
        # biotite's writer trips a numpy DeprecationWarning on every call. Suppressed here
        # rather than globally: importing atomworks resets the filters.
        warnings.simplefilter("ignore", DeprecationWarning)
        f.set_structure(atom_array)
    buf = io.StringIO()
    f.write(buf)
    return buf.getvalue()


target = aa[tgt]
target = target[~np.isnan(target.coord).any(axis=-1)]

view = py3Dmol.view(width=640, height=420)
view.addModel(to_pdb(target), "pdb")
view.setStyle({"cartoon": {"color": "0xb9c0c9"}})
view.addStyle({"resi": HOTSPOTS}, {"stick": {"colorscheme": "orangeCarbon", "radius": 0.3}})
view.setBackgroundColor("0xffffff")
view.zoomTo()
view.show()

## 3. Generate

PPI settings per the [recycling schedule table](../how-to/run-the-model.md#recycling-schedules).

In [ ]:
NUM_REPLICATES = int(os.environ.get("CYCLIC_BINDER_REPLICATES", 4))


def generate(chemistry):
    out_dir = WORK / chemistry
    if (out_dir / "inference_samples").exists():
        print(f"{chemistry}: reusing {out_dir}")
        return out_dir
    cmd = [
        "rfd4-generate",
        f"inputs={INPUTS / f'01_PD1_{chemistry}_{LENGTH:02d}.cif.zst'}",
        f"num_replicates={NUM_REPLICATES}",
        f"out_dir={out_dir}",
        "seed=0",
        "batch_size=1",
        "recycle_strategy=medium/front_loaded",
        "compress_outputs=false",
    ]
    print(" ".join(cmd))
    subprocess.run(cmd, cwd=REPO_ROOT, check=True)
    return out_dir


RUNS = {c: generate(c) for c in CHEMISTRIES}

## 4. Closure and binding

Two separate questions: a ring that closes in empty space is not a binder, and a peptide
packed against the hotspots with an open ring is not a macrocycle. Failures stay in the
denominator.

The closure test is **two-sided**. A one-sided "under 2.3 A" cut scores an impossible
overshoot as success — an un-atomized disulfide can land near 1.2 A, inside a single sulfur.

In [ ]:
import biotite.structure as struc
import biotite.structure.io.pdbx as pdbx

# (atom on residue 1, atom on residue LENGTH, ideal distance, tolerance)
CLOSURE = {
    "head_tail":  ("N",  "C",  1.33, 0.30),
    "disulfide":  ("SG", "SG", 2.05, 0.30),
    "isopeptide": ("NZ", "CG", 1.33, 0.30),
}


def designs(run_dir):
    return sorted(glob.glob(str(Path(run_dir) / "inference_samples" / "**" / "*.cif"), recursive=True))


def measure(path, chemistry):
    arr = pdbx.get_structure(pdbx.CIFFile.read(path), model=1)
    arr = arr[struc.filter_amino_acids(arr)]
    sizes = {c: int((arr.chain_id == c).sum()) for c in np.unique(arr.chain_id)}
    pep_chain, tgt_chain = min(sizes, key=sizes.get), max(sizes, key=sizes.get)
    p, t = arr[arr.chain_id == pep_chain], arr[arr.chain_id == tgt_chain]

    atom_a, atom_b, ideal, tol = CLOSURE[chemistry]

    def coord(sub, res_id, name):
        sel = (sub.res_id == res_id) & (sub.atom_name == name)
        return sub.coord[sel][0] if sel.sum() == 1 else None

    a, b = coord(p, 1, atom_a), coord(p, LENGTH, atom_b)
    d = float(np.linalg.norm(a - b)) if a is not None and b is not None else np.nan

    hs = t[np.isin(t.res_id, HOTSPOTS)]
    dm = np.linalg.norm(p.coord[:, None, :] - hs.coord[None, :, :], axis=-1)
    # Count hotspot *residues* in contact, not their atoms: one residue contributes many atoms.
    engaged = len({int(r) for r, dd in zip(hs.res_id, dm.min(axis=0)) if dd <= 5.0})
    return d, bool(np.isfinite(d) and abs(d - ideal) <= tol), engaged


print(f"{'chemistry':<12}{'closed':>9}{'median':>9}{'range':>16}{'hotspots engaged':>19}")
for c in CHEMISTRIES:
    rows = [measure(p, c) for p in designs(RUNS[c])]
    d = np.array([r[0] for r in rows]); fin = d[np.isfinite(d)]
    n_closed = sum(r[1] for r in rows)
    eng = np.median([r[2] for r in rows])
    rng = f"{fin.min():.2f}-{fin.max():.2f}" if len(fin) else "n/a"
    med = f"{np.median(fin):.3f}" if len(fin) else "n/a"
    print(f"{c:<12}{n_closed:>5}/{len(rows):<3}{med:>9}{rng:>16}{eng:>13.0f} of {len(HOTSPOTS)}")

## 5. The designs

Target grey, hotspots orange, macrocycle blue, closure bond as a green dashed line.

In [ ]:
def show(chemistry, index=0):
    path = designs(RUNS[chemistry])[index]
    arr = pdbx.get_structure(pdbx.CIFFile.read(path), model=1)
    arr = arr[struc.filter_amino_acids(arr)]
    sizes = {c: int((arr.chain_id == c).sum()) for c in np.unique(arr.chain_id)}
    pep_chain, tgt_chain = min(sizes, key=sizes.get), max(sizes, key=sizes.get)
    p = arr[arr.chain_id == pep_chain]

    atom_a, atom_b, ideal, _ = CLOSURE[chemistry]
    ca = p.coord[(p.res_id == 1) & (p.atom_name == atom_a)]
    cb = p.coord[(p.res_id == LENGTH) & (p.atom_name == atom_b)]

    view = py3Dmol.view(width=680, height=460)
    view.addModel(to_pdb(arr[arr.chain_id == tgt_chain]), "pdb")
    view.setStyle({"cartoon": {"color": "0xb9c0c9"}})
    view.addStyle({"resi": HOTSPOTS}, {"stick": {"colorscheme": "orangeCarbon", "radius": 0.3}})

    view.addModel(to_pdb(p), "pdb")
    view.setStyle({"model": -1}, {"cartoon": {"color": "0x3b82f6"},
                                  "stick": {"colorscheme": "cyanCarbon", "radius": 0.15}})
    if len(ca) and len(cb):
        d = float(np.linalg.norm(ca[0] - cb[0]))
        view.addLine({"start": dict(zip("xyz", map(float, ca[0]))),
                      "end": dict(zip("xyz", map(float, cb[0]))),
                      "color": "0x16a34a", "dashed": True, "linewidth": 4})
        print(f"{chemistry}: closure {atom_a}1-{atom_b}{LENGTH} = {d:.3f} A (ideal {ideal})")
    view.setBackgroundColor("0xffffff")
    view.zoomTo({"model": -1})
    view.show()


show("head_tail")

In [ ]:
show("disulfide")

In [ ]:
show("isopeptide")

## 6. Results

16 replicates per chemistry, seed 0, released checkpoint, one H200. Closed means
`|d - ideal| <= 0.3 A`.

| Chemistry | Closed | Median (A) | Range (A) | Hotspots |
|---|---|---|---|---|
| head-to-tail | **16/16** | 1.318 | 1.277-1.390 | 4 of 4 |
| disulfide | 12/16 | 2.119 | 1.576-2.724 | 4 of 4 |
| isopeptide | **1/16** | 2.136 | 1.478-4.412 | 4 of 4 |

**All three dock.** Every chemistry engages all four hotspots with a comparable interface
(114-164 atom contacts within 5 A). Closure is the only differentiator.

**Head-to-tail is production-ready** — 16/16 at 1.318 A, spread 0.11 A. Over lengths 6-20 it
closes 360/360 at median 1.308 A, with no length dependence.

**Disulfide needs replicate filtering** — 12/16 here, 177/360 over lengths 6-20, best at
10-16 and failing at 6-7 where the ring cannot take the bridge. The median sits on the ideal,
so the problem is spread, not aim.

**Isopeptide does not work de novo** — 1/16 here, 18/360 over lengths 6-20. With input
coordinates it is reliable (24/24 at 1.363 A, atomized), so this is the regime, not the
chemistry. In the length sweep the model usually closed onto the **wrong atom**: `LYS.NZ`
against the carboxyl oxygen `OD1` (median 1.614 A, closest contact in 243/360) rather than
the `NZ`-`CG` amide (2.325 A). A threshold on the declared pair alone cannot tell that from a
ring that never closed, which is why closure is reported per atom pair.

No refolding oracle was run: these are geometries, not designability claims.